# PlanWise gateways on Kaggle: Laya + LLM (Ollama), one run, one token

Settings: Accelerator = GPU T4 x2, Internet = On. Run cell 1, then cell 2, and **leave cell 2 running** (stopping it closes the links).
Cell 2 prints ready-to-paste lines for `Backend/.env`. Same token for both, two URLs. Do not share the output.
Safe to re-run cell 2 after an interrupt: it stops the old servers first, so the printed token always matches.
Generated from `Backend/laya_server/server.py` and `Backend/llm_server/server.py`.


In [ ]:
import os, shutil, subprocess

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q laya fastapi uvicorn requests")
sh("apt-get update -qq && apt-get install -y -qq zstd pciutils")   # zstd: Ollama installer; pciutils: GPU detection
if subprocess.run("nvidia-smi -L", shell=True).returncode != 0:
    raise SystemExit("No GPU visible. Settings > Accelerator > GPU T4 x2, then restart the session.")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh")
cf = "/tmp/cloudflared"
if not os.path.exists(cf):
    sh(f"curl -fsSL -o {cf} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {cf}")
print("setup done")


In [ ]:
open("laya_app.py", "w").write('"""Laya inference server. Runs on Kaggle (NOT on the developer\'s computer), behind a Cloudflare tunnel.\n\nPOST /analyze  (Authorization: Bearer $LAYA_TOKEN)  {"headlines": [...]}\nGET  /health   (no auth)\n"""\nimport hmac\nimport os\nimport threading\n\nos.environ.setdefault("USE_TF", "0")  # model card: avoids a TensorFlow deadlock when loading Laya\n\nimport laya\nfrom fastapi import FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nTOKEN = os.environ["LAYA_TOKEN"]\nMODEL = os.environ.get("LAYA_MODEL", "convaiinnovations/laya")\nagent = laya.load(MODEL)\nlock = threading.Lock()  # one prediction at a time\n\nQUESTIONS = {\n    "sentiment": {\n        "type": "choice",\n        "instructions": "What is the likely effect of this financial news on the stock price?",\n        "criteria": {\n            "bearish": "negative for the stock price",\n            "bullish": "positive for the stock price",\n            "neutral": "no clear effect, or purely informational",\n        },\n    },\n    "kind": {\n        "type": "choice",\n        "instructions": "Is this headline a concrete company event, or generic filler?",\n        "criteria": {\n            "event": "reports a concrete event: earnings, guidance, a deal, a rating change, a product or legal or regulatory action involving the company",\n            "filler": "generic explainer, stock-quote page, promotion, or fund-holding boilerplate",\n        },\n    },\n}\n\n\nclass Req(BaseModel):\n    headlines: list[str] = Field(max_length=200)\n\n\napp = FastAPI(title="Laya server")\n\n\n@app.get("/health")\ndef health():\n    return {"ok": True, "model": MODEL}\n\n\n@app.post("/analyze")\ndef analyze(req: Req, authorization: str | None = Header(default=None)):\n    if not authorization or not hmac.compare_digest(authorization, f"Bearer {TOKEN}"):\n        raise HTTPException(401, "bad token")\n    results = []\n    with lock:\n        for h in req.headlines:\n            a = agent.predict(h[:500], QUESTIONS)["answers"]\n            results.append({\n                "sentiment": a["sentiment"]["choice"],\n                "sentiment_probs": a["sentiment"]["probabilities"],\n                "kind": a["kind"]["choice"],\n                "kind_probs": a["kind"]["probabilities"],\n            })\n    return {"model": MODEL, "results": results}\n')
open("llm_app.py", "w").write('"""LLM gateway. Runs on Kaggle (NOT on the developer\'s computer), in front of a local Ollama, behind a Cloudflare tunnel.\n\nPOST /chat    (Authorization: Bearer $LLM_TOKEN)  {"messages": [...], "format": <json schema>, "options": {...}}\nGET  /health  (no auth)\n\nThin on purpose: prompts, schemas and validation live in Backend/pipeline/llm.py so they are versioned with the repo.\n"""\nimport hmac\nimport os\n\nimport requests\nfrom fastapi import FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nTOKEN = os.environ["LLM_TOKEN"]\nMODEL = os.environ.get("LLM_MODEL", "qwen3:8b")\nOLLAMA = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")\n\n\nclass Req(BaseModel):\n    messages: list[dict] = Field(max_length=20)\n    format: dict | str | None = None   # JSON schema, or "json"\n    options: dict | None = None        # e.g. {"temperature": 0, "num_ctx": 8192}\n    think: bool = False                # qwen3 reasoning mode; off by default\n\n\napp = FastAPI(title="LLM gateway")\n\n\n@app.get("/health")\ndef health():\n    return {"ok": True, "model": MODEL}\n\n\n@app.post("/chat")\ndef chat(req: Req, authorization: str | None = Header(default=None)):\n    if not authorization or not hmac.compare_digest(authorization, f"Bearer {TOKEN}"):\n        raise HTTPException(401, "bad token")\n    body = {"model": MODEL, "messages": req.messages, "stream": False, "think": req.think}\n    if req.format:\n        body["format"] = req.format\n    if req.options:\n        body["options"] = req.options\n    r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=280)\n    if r.status_code != 200:\n        raise HTTPException(502, f"ollama: {r.text[:300]}")\n    j = r.json()\n    return {\n        "model": MODEL,\n        "content": j["message"]["content"],\n        "tokens_in": j.get("prompt_eval_count"),\n        "tokens_out": j.get("eval_count"),\n    }\n')
print("server files written")


In [ ]:
import os, re, secrets, subprocess, sys, time, urllib.request

MODEL = "qwen3:8b"   # ~5-6 GB in VRAM at Q4
OLLAMA = "http://127.0.0.1:11434"
cf = "/tmp/cloudflared"

def up(url):
    try:
        urllib.request.urlopen(url, timeout=5)
        return True
    except Exception:
        return False

def wait(url, what, limit):
    t0 = time.time()
    while not up(url):
        if time.time() - t0 > limit:
            raise SystemExit(f"{what} did not start in {limit}s")
        time.sleep(3)
    print(f"{what} up after {time.time() - t0:.0f}s")

# fresh start every time, so the token printed below is the one the servers use
subprocess.run("pkill -f 'uvicorn laya_app' ; pkill -f 'uvicorn llm_app' ; pkill -f cloudflared", shell=True)
time.sleep(2)

TOKEN = secrets.token_urlsafe(24)   # one new random password per run; the tunnel URLs are public, this protects them

# Ollama on GPU 0 (kept if already running), Laya on GPU 1
if not up(OLLAMA + "/api/tags"):
    subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", CUDA_VISIBLE_DEVICES="0"),
                     stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    wait(OLLAMA + "/api/tags", "ollama", 60)
subprocess.run(["ollama", "pull", MODEL], check=True)
subprocess.run(["ollama", "ps"])

base = dict(os.environ, LLM_TOKEN=TOKEN, LAYA_TOKEN=TOKEN, LLM_MODEL=MODEL, USE_TF="0")
servers = {
    "LLM": subprocess.Popen([sys.executable, "-m", "uvicorn", "llm_app:app", "--host", "127.0.0.1", "--port", "8766"],
                            env=dict(base, CUDA_VISIBLE_DEVICES="0"), stdout=open("llm.log", "w"), stderr=subprocess.STDOUT),
    "LAYA": subprocess.Popen([sys.executable, "-m", "uvicorn", "laya_app:app", "--host", "127.0.0.1", "--port", "8765"],
                             env=dict(base, CUDA_VISIBLE_DEVICES="1"), stdout=open("laya.log", "w"), stderr=subprocess.STDOUT),
}
for name, port, log in (("LLM", 8766, "llm.log"), ("LAYA", 8765, "laya.log")):
    try:
        wait(f"http://127.0.0.1:{port}/health", name + " gateway", 600)
    except SystemExit:
        print(open(log).read()[-3000:])
        raise
    if servers[name].poll() is not None:
        print(open(log).read()[-3000:])
        raise SystemExit(name + " gateway crashed (log above)")

tunnels, urls = {}, {}
for name, port in (("LLM", 8766), ("LAYA", 8765)):
    tunnels[name] = subprocess.Popen([cf, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                     stdout=open(f"tunnel_{name}.log", "w"), stderr=subprocess.STDOUT)
for name in tunnels:
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"tunnel_{name}.log").read())
        if m:
            urls[name] = m.group(0)
            break
        time.sleep(2)
    else:
        print(open(f"tunnel_{name}.log").read()[-2000:])
        raise SystemExit(f"no tunnel URL for {name}")
    for _ in range(30):   # wait until the public URL actually answers
        if up(urls[name] + "/health"):
            break
        time.sleep(2)

print("=" * 60)
print("Paste these 4 lines into Backend/.env (replace the old ones):")
print("LLM_URL=" + urls["LLM"])
print("LLM_TOKEN=" + TOKEN)
print("LAYA_URL=" + urls["LAYA"])
print("LAYA_TOKEN=" + TOKEN)
print("=" * 60)
print("Leave this cell running. Stopping it closes the links.")

end = time.time() + 11 * 3600
beat = 0
while time.time() < end:
    dead = [n for n, p in {**servers, **{t + "-tunnel": q for t, q in tunnels.items()}}.items() if p.poll() is not None]
    if dead:
        print("died:", dead, "- re-run this cell")
        break
    time.sleep(60)
    beat += 1
    if beat % 30 == 0:
        print(f"still serving, {beat // 60}h{beat % 60:02d}m")
